In [ ]:
%load_ext autoreload
%autoreload 2

import humanoid_bench

from fast_td3.actors import ActorEGNN, Actor, ActorMPNN, ActorHEPI
from fast_td3.actors.gnn.aegnn import AngleEGNN, get_edges_batch
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

import torch

from fast_td3.actors.gnn.egnn import unsorted_segment_sum

In [ ]:
num_envs = 16

envs = HumanoidBenchEnv("h1-maze-v0", num_envs, device="cuda:0")

obs, xanchor = envs.reset()

n_act = envs.num_actions
n_obs = envs.num_obs if type(envs.num_obs) == int else envs.num_obs[0]

In [ ]:
# Initialize EGNN
egnn = ActorEGNN(n_obs = n_obs, n_act=n_act, num_envs=num_envs, init_scale=0.1, hidden_dim=32, act_fn="relu", device="cuda:0", n_layers=4, robot="h1", n_edge_feat=0, coords_agg="sum", batch_size=8192)

import time
start = time.time()
for _ in range(1000):
    # Forward pass through EGNN
    egnn.forward(obs, xanchor)
end = time.time()
print(f"Forward pass time: {end - start:.6f} seconds")

In [ ]:
actor = ActorEGNN(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=112,
    n_layers=2,
    init_scale=0.1,
    act_fn="relu"
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xanchor)

In [ ]:
actor = ActorMPNN(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=96,
    latent_dim=96,
    num_message=2,
    node_encoder_layers=2,
    edge_encoder_layers=2,
    node_decoder_layers=2,
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xanchor)

In [ ]:
actor = ActorHEPI(
    n_obs=19,
    n_act=19,
    num_envs=4,
    device="cuda:0",
    batch_size=4,
    n_node_feat=2,
    n_edge_feat=1,
    hidden_dim=64,
    latent_dim=64,
    num_message=2,
    node_encoder_layers=2,
    edge_encoder_layers=2,
    node_decoder_layers=2,
)

print(f"Actor num of parameters: {sum(p.numel() for p in actor.parameters())}")

#result = actor.explore(obs, xanchor)

In [ ]:
import torch
from torch_geometric.data import HeteroData
from collections import defaultdict


class H1:
    joint_dict = {
        "left_hip_yaw": 0,
        "left_hip_roll": 1,
        "left_hip_pitch": 2,
        "left_knee": 3,
        "left_ankle": 4,
        "right_hip_yaw": 5,
        "right_hip_roll": 6,
        "right_hip_pitch": 7,
        "right_knee": 8,
        "right_ankle": 9,
        "torso": 10,
        "left_shoulder_pitch": 11,
        "left_shoulder_roll": 12,
        "left_shoulder_yaw": 13,
        "left_elbow": 14,
        "right_shoulder_pitch": 15,
        "right_shoulder_roll": 16,
        "right_shoulder_yaw": 17,
        "right_elbow": 18,
    }

    edge_list = [
        (joint_dict["left_hip_yaw"], joint_dict["left_hip_roll"]),
        (joint_dict["left_hip_roll"], joint_dict["left_hip_pitch"]),
        (joint_dict["left_hip_pitch"], joint_dict["left_knee"]),
        (joint_dict["left_knee"], joint_dict["left_ankle"]),
        (joint_dict["right_hip_yaw"], joint_dict["right_hip_roll"]),
        (joint_dict["right_hip_roll"], joint_dict["right_hip_pitch"]),
        (joint_dict["right_hip_pitch"], joint_dict["right_knee"]),
        (joint_dict["right_knee"], joint_dict["right_ankle"]),
        (joint_dict["torso"], joint_dict["left_hip_yaw"]),
        (joint_dict["torso"], joint_dict["right_hip_yaw"]),
        (joint_dict["torso"], joint_dict["left_shoulder_pitch"]),
        (joint_dict["left_shoulder_pitch"], joint_dict["left_shoulder_roll"]),
        (joint_dict["left_shoulder_roll"], joint_dict["left_shoulder_yaw"]),
        (joint_dict["left_shoulder_yaw"], joint_dict["left_elbow"]),
        (joint_dict["torso"], joint_dict["right_shoulder_pitch"]),
        (joint_dict["right_shoulder_pitch"], joint_dict["right_shoulder_roll"]),
        (joint_dict["right_shoulder_roll"], joint_dict["right_shoulder_yaw"]),
        (joint_dict["right_shoulder_yaw"], joint_dict["right_elbow"]),
    ]

    edge_type_encoding = [
        0, 1, 2, 3,
        0, 1, 2, 3,
        4, 4,
        5, 6, 7, 8,
        5, 6, 7, 8,
    ]

    num_nodes = len(joint_dict)
    num_edges = len(edge_list)


# === Step 1: Joint type mapping ===
JOINT_TYPE_MAP = {
    "left_hip_yaw": "hip",
    "left_hip_roll": "hip",
    "left_hip_pitch": "hip",
    "right_hip_yaw": "hip",
    "right_hip_roll": "hip",
    "right_hip_pitch": "hip",
    "left_knee": "knee",
    "right_knee": "knee",
    "left_ankle": "ankle",
    "right_ankle": "ankle",
    "torso": "torso",
    "left_shoulder_pitch": "shoulder",
    "left_shoulder_roll": "shoulder",
    "left_shoulder_yaw": "shoulder",
    "left_elbow": "elbow",
    "right_shoulder_pitch": "shoulder",
    "right_shoulder_roll": "shoulder",
    "right_shoulder_yaw": "shoulder",
    "right_elbow": "elbow",
}


# === Step 2: Generate dummy features ===
angles = torch.randn(H1.num_nodes, 1)      # Replace with real angles
velocities = torch.randn(H1.num_nodes, 1)  # Replace with real velocities
node_features = torch.cat([angles, velocities], dim=1)  # Shape: [19, 2]


# === Step 3: Build HeteroData ===
data = HeteroData()
joint_names = list(H1.joint_dict.keys())
node_type_to_entries = defaultdict(list)  # {node_type: [(global_idx, joint_name)]}

# Group nodes by type
for joint_name, global_idx in H1.joint_dict.items():
    joint_type = JOINT_TYPE_MAP[joint_name]
    node_type_to_entries[joint_type].append((global_idx, joint_name))

# Add nodes by type
global_to_local = {}  # Maps: global_idx → (node_type, local_idx)
for node_type, entries in node_type_to_entries.items():
    indices = [idx for idx, _ in entries]
    feats = node_features[indices]  # [num_nodes_of_type, 2]
    data[node_type].x = feats
    for local_idx, (global_idx, _) in enumerate(entries):
        global_to_local[global_idx] = (node_type, local_idx)

# === Step 4: Add typed edges ===
for (src_global, dst_global), edge_type in zip(H1.edge_list, H1.edge_type_encoding):
    src_type, src_local = global_to_local[src_global]
    dst_type, dst_local = global_to_local[dst_global]
    edge_index = torch.tensor([[src_local], [dst_local]], dtype=torch.long)
    rel_name = f'edge_type_{edge_type}'
    data[(src_type, rel_name, dst_type)].edge_index = edge_index


# === Optional: Sanity check ===
print(data)
for ntype in data.node_types:
    print(f"Node type '{ntype}' has {data[ntype].x.shape[0]} nodes")

for etype in data.edge_types:
    print(f"Edge type {etype}: {data[etype].edge_index.shape[1]} edges")


In [ ]:
# Example: How unsorted_segment_sum works
import torch
from fast_td3.actors.gnn.egnn import unsorted_segment_sum

print("=== Example: unsorted_segment_sum ===")
print()

# Example data: 6 nodes with 3 features each
data = torch.tensor([
    [1.0, 2.0, 3.0],    # Node 0
    [4.0, 5.0, 6.0],    # Node 1  
    [7.0, 8.0, 9.0],    # Node 2
    [10.0, 11.0, 12.0], # Node 3
    [13.0, 14.0, 15.0], # Node 4
    [16.0, 17.0, 18.0]  # Node 5
])

# Segment IDs: which group each node belongs to
# Group 0: nodes 0, 2, 4
# Group 1: nodes 1, 5  
# Group 2: node 3
segment_ids = torch.tensor([0, 1, 0, 2, 0, 1])

num_segments = 3

print("Input data:")
print(data)
print(f"Shape: {data.shape}")
print()

print("Segment IDs:")
print(segment_ids)
print("Group assignments:")
print("  Group 0: nodes 0, 2, 4")
print("  Group 1: nodes 1, 5") 
print("  Group 2: node 3")
print()

# Apply unsorted_segment_sum
result = unsorted_segment_sum(data, segment_ids, num_segments)

print("Result (sum within each group):")
print(result)
print(f"Shape: {result.shape}")
print()

print("Manual verification:")
print("Group 0 sum:", data[0] + data[2] + data[4])
print("Group 1 sum:", data[1] + data[5])
print("Group 2 sum:", data[3])
print()

print("Use case in GNNs:")
print("- data: node features or edge features")
print("- segment_ids: which edges belong to which target node")
print("- result: aggregated features for each target node")

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
from fast_td3.robots.h1 import h1

id_to_joint = {v: k for k, v in h1.joint_dict.items()}

G = nx.DiGraph()
G.add_edges_from(h1.edge_list)
G = nx.relabel_nodes(G, id_to_joint)

# Bigger figure
plt.figure(figsize=(16, 12))

# Increase k to spread nodes further apart
pos = nx.spring_layout(G, k=2.0, iterations=300, seed=42)

# Draw nodes
nx.draw_networkx_nodes(G, pos, node_color="skyblue", node_size=1500)

# Draw edges
nx.draw_networkx_edges(G, pos, edge_color="gray", arrows=True, arrowsize=15, width=1.2)

# Draw labels with larger font
nx.draw_networkx_labels(G, pos, font_size=10, font_weight="bold")

plt.axis("off")
plt.title("Robot Joint Connectivity Graph", fontsize=16)
plt.tight_layout()
plt.show()
